# Task 6 — Feature Selection and Recommendation Quality Evaluation

**Objective:** compare four feature sets to understand how the selected text fields affect recommendations.

1. Title only
2. Title + Tags
3. Title + Description + Tags
4. Title + Description + Transcript + Tags

This notebook produces recommendation lists for five talks. It does not claim that one feature set is objectively best: there is no labelled ground-truth relevance dataset in the supplied CSV, so the final quality judgement requires manual review.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the project folder from the current Jupyter working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "ted_talks_merged.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ted_talks_merged.csv. "
        "Open Jupyter from the TED_Talks_Recommendation_Ajay project folder."
    )


PROJECT_DIR = find_project_root()
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "ted_talks_merged.csv"
PROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"
print("Project folder:", PROJECT_DIR)

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError("Run Task 3 first.")
df = pd.read_csv(PROCESSED_DATA_PATH)
for column in ["title_clean", "tags_clean", "description_clean", "transcript_clean"]:
    if column not in df.columns:
        df[column] = ""
    df[column] = df[column].fillna("").astype(str)

df["title"] = df["title"].fillna("").astype(str)
feature_sets = {
    "1 - Title only": ["title_clean"],
    "2 - Title + Tags": ["title_clean", "tags_clean"],
    "3 - Title + Description + Tags": ["title_clean", "description_clean", "tags_clean"],
    "4 - Title + Description + Transcript + Tags": ["title_clean", "description_clean", "transcript_clean", "tags_clean"],
}

# Use five actual titles from the dataset. Replace them with other titles for your review if needed.
sample_titles = df["title"].drop_duplicates().head(5).tolist()
print("Selected test talks:")
for title in sample_titles:
    print("-", title)

Project folder: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay
Selected test talks:
- Do schools kill creativity?
- Averting the climate crisis
- Simplicity sells
- Greening the ghetto
- The best stats you've ever seen


## 1. Fit one vectorizer per feature combination

In [3]:
models = {}
for feature_name, columns in feature_sets.items():
    text = df[columns].fillna("").agg(" ".join, axis=1)
    vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2), min_df=2)
    matrix = vectorizer.fit_transform(text)
    models[feature_name] = {"vectorizer": vectorizer, "matrix": matrix, "text": text}
    print(feature_name, "-> matrix shape:", matrix.shape)

1 - Title only -> matrix shape: (2550, 1562)
2 - Title + Tags -> matrix shape: (2550, 4507)
3 - Title + Description + Tags -> matrix shape: (2550, 14112)
4 - Title + Description + Transcript + Tags -> matrix shape: (2550, 20000)


## 2. Generate top-five recommendations for five talks

In [4]:
def recommend_for_feature_set(feature_name, selected_title, top_n=5):
    model = models[feature_name]
    matches = df.index[df["title"].str.casefold() == selected_title.casefold()].tolist()
    if not matches:
        raise ValueError(f"Title not found: {selected_title}")
    selected_index = matches[0]
    scores = cosine_similarity(model["matrix"][selected_index], model["matrix"]).ravel()
    scores[selected_index] = -1
    best = np.argsort(scores)[::-1][:min(top_n, len(df)-1)]
    result = df.iloc[best][[column for column in ["talk_id", "title", "main_speaker", "url"] if column in df.columns]].copy()
    result["similarity_score"] = scores[best]
    result.insert(0, "selected_talk", selected_title)
    result.insert(1, "feature_set", feature_name)
    return result.reset_index(drop=True)

results = []
for feature_name in feature_sets:
    for selected_title in sample_titles:
        results.append(recommend_for_feature_set(feature_name, selected_title, top_n=5))
comparison_df = pd.concat(results, ignore_index=True)
print("Total recommendation rows:", len(comparison_df))
display(comparison_df)

Total recommendation rows: 100


,selected_talk,feature_set,talk_id,title,main_speaker,url,similarity_score
0,Do schools kill creativity?,1 - Title only,Linda Hill: How to manage for collective creat...,How to manage for collective creativity,Linda Hill,https://www.ted.com/talks/linda_hill_how_to_ma...,0.552639
1,Do schools kill creativity?,1 - Title only,Julie Burstein: 4 lessons in creativity,4 lessons in creativity,Julie Burstein,https://www.ted.com/talks/julie_burstein_4_les...,0.408704
2,Do schools kill creativity?,1 - Title only,Bill Gates: How state budgets are breaking US ...,How state budgets are breaking US schools,Bill Gates,https://www.ted.com/talks/bill_gates_how_state...,0.407503
3,Do schools kill creativity?,1 - Title only,Geoffrey Canada: Our failing schools. Enough i...,Our failing schools. Enough is enough!,Geoffrey Canada,https://www.ted.com/talks/geoffrey_canada_our_...,0.395717
4,Do schools kill creativity?,1 - Title only,Suzanne Talhouk: Don't kill your language,Don't kill your language,Suzanne Talhouk,https://www.ted.com/talks/suzanne_talhouk_don_...,0.385514
...,...,...,...,...,...,...,...
95,The best stats you've ever seen,4 - Title + Description + Transcript + Tags,Hans Rosling: Let my dataset change your mindset,Let my dataset change your mindset,Hans Rosling,https://www.ted.com/talks/hans_rosling_at_state,0.551549
96,The best stats you've ever seen,4 - Title + Description + Transcript + Tags,Hans Rosling: New insights on poverty,New insights on poverty,Hans Rosling,https://www.ted.com/talks/hans_rosling_reveals...,0.399930
97,The best stats you've ever seen,4 - Title + Description + Transcript + Tags,Hans Rosling: The good news of the decade? We'...,The good news of the decade? We're winning the...,Hans Rosling,https://www.ted.com/talks/hans_rosling_the_goo...,0.343110
98,The best stats you've ever seen,4 - Title + Description + Transcript + Tags,Hans Rosling: Religions and babies,Religions and babies,Hans Rosling,https://www.ted.com/talks/hans_rosling_religio...,0.322849


## 3. Compare how the lists change

Use this table to inspect whether each feature combination returns semantically relevant talks. Review at least five selected talks and compare all four feature sets.

In [5]:
# Compact table of titles only, grouped by feature set and selected talk.
compact = comparison_df.groupby(["selected_talk", "feature_set"])["title"].apply(list).reset_index(name="top_5_titles")
display(compact)

,selected_talk,feature_set,top_5_titles
0,Averting the climate crisis,1 - Title only,"[New thinking on the climate crisis, The scien..."
1,Averting the climate crisis,2 - Title + Tags,"[New thinking on the climate crisis, The scien..."
2,Averting the climate crisis,3 - Title + Description + Tags,"[New thinking on the climate crisis, What come..."
3,Averting the climate crisis,4 - Title + Description + Transcript + Tags,"[New thinking on the climate crisis, A climate..."
4,Do schools kill creativity?,1 - Title only,"[How to manage for collective creativity, 4 le..."
5,Do schools kill creativity?,2 - Title + Tags,"[Authentic creativity vs. karaoke culture, Kid..."
6,Do schools kill creativity?,3 - Title + Description + Tags,"[Bring on the learning revolution!, How to esc..."
7,Do schools kill creativity?,4 - Title + Description + Transcript + Tags,"[How to escape education's death valley, Bring..."
8,Greening the ghetto,1 - Title only,"[Do schools kill creativity?, How a video game..."
9,Greening the ghetto,2 - Title + Tags,"[Winning the oil endgame, Rebuilding a neighbo..."


## 4. Manual evaluation notes

For each feature set, record whether the five recommendations are relevant, whether any unrelated talk appears, and whether transcript text helps or adds noise.

| Feature set | What to check |
|---|---|
| Title only | Does a short title capture enough context? |
| Title + Tags | Do tags improve topic matching? |
| Title + Description + Tags | Does the summary improve the match? |
| Title + Description + Transcript + Tags | Does the transcript improve relevance, or does it introduce repeated/general words? |

**Important:** similarity scores from different vectorizers are not a direct human-quality metric. The notebook cannot select a best feature combination without a judged evaluation set. Record your own observations after reviewing the printed lists.